# convtranspose-bn-activation-block — worked example 3: Stack two ConvT+BN+ReLU blocks to 4x upsample

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `convtranspose-bn-activation-block`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A DCGAN generator chains several `ConvT -> BN -> ReLU` blocks, each doubling spatial size and (typically) halving channels. Stacking two such blocks multiplies the spatial size by 4 while taking the channel count e.g. 256 -> 128 -> 64. Composing them in a single `nn.Sequential` lets the whole sub-network be called as one module.

## Worked solution

**Step 1 — Define the reusable block.** Write a helper that returns one canonical block: `ConvTranspose2d(kernel=4, stride=2, padding=1, bias=False) -> BatchNorm2d -> ReLU(inplace=True)`. This is the repeated unit.

**Step 2 — Chain channel counts correctly.** The first block maps `256 -> 128`, the second maps `128 -> 64`. The `out_channels` of block one must equal the `in_channels` of block two, or the tensors won't line up.

**Step 3 — Each block doubles spatial size.** Starting at 4x4: block one -> 8x8, block two -> 16x16. Two doublings is a 4x increase, so a 4x4 input ends at 16x16.

**Step 4 — Compose into one Sequential.** Put both blocks inside an outer `nn.Sequential(*blocks)`. Flatten by unpacking the inner Sequentials' children, or simply nest them — calling the outer module runs them in order. Running a `(1, 256, 4, 4)` input must yield `(1, 64, 16, 16)`.

In [ ]:
import torch.nn as nn

def _block(in_c, out_c):
    return nn.Sequential(
        nn.ConvTranspose2d(in_c, out_c, kernel_size=4, stride=2, padding=1, bias=False),
        nn.BatchNorm2d(out_c),
        nn.ReLU(inplace=True),
    )

def build_two_block_generator():
    return nn.Sequential(
        _block(256, 128),
        _block(128, 64),
    )

t.manual_seed(0)
net = build_two_block_generator()
x = t.randn(1, 256, 4, 4)
out = net(x)
print('output shape:', tuple(out.shape))
n_convt = sum(isinstance(m, nn.ConvTranspose2d) for m in net.modules())
print('num convtranspose layers:', n_convt)